# MAGNET: African Tokenization Fairness — Complete Colab Training Pipeline

## Overview

**Project goal:** this notebook is part of a project reproducing and extending MAGNET (Ahia et al., NeurIPS 2024) — a gradient-based tokenization method — to evaluate tokenization fairness across 9 African languages (Swahili, Zulu, Yoruba, Igbo, Hausa, Chichewa/Nyanja, Amharic, Kinyarwanda, Wolof). See the repo's `README.md` and `data_card.md` for full background.

**What this notebook does**, end to end, on a Colab GPU runtime:
1. Mounts Google Drive (where the training corpora and checkpoints live).
2. Clones this repo and installs its dependencies.
3. Runs MAGNET training with per-language boundary-predictor routing and a uniform target compression rate (β=0.5 for every language) — the project's validated, training-stable baseline.
4. Inspects the trained checkpoint's segmentation behavior and reports the fairness metrics used throughout this project.
5. Ends with a clearly-separated **Known limitation** appendix reproducing a documented training failure (per-language-tuned β collapsing under the boundary-predictor regularizer), kept here for transparency rather than presented as a working pipeline.

This notebook does not implement any model or data-loading logic itself — it only drives the code already in `src/model/`, `src/training/`, and `src/eval/`.

## Prerequisites

- **Google Drive access** with a `DATA_ROOT` folder containing the 9 languages' corpora and `eval_holdout/` set (see `data_card.md` for the exact structure). This notebook expects it at `/content/drive/MyDrive/DATA_ROOT` — adjust the paths below if yours lives elsewhere.
- **GitHub access to this repo** — it's public (`mohamad-755/magnet-african-tokenization`), so a plain `git clone` works with no authentication or personal access token needed.
- **A GPU runtime** — `Runtime > Change runtime type > T4 GPU` (or better) in the Colab menu, before running any cells below. Training will be extremely slow on CPU.

### Why mount Drive

Colab's local disk is wiped on every disconnect, but the training corpora are large (Hausa's alone is 252MB) and checkpoints need to persist across sessions. Mounting Drive lets training read `DATA_ROOT` directly and write checkpoints to a path that survives a disconnect — pointing `--checkpoint-dir` anywhere else means losing all progress if the runtime resets.

In [ ]:
# Mount Google Drive — DATA_ROOT (corpora) and checkpoint output both live here,
# since Colab's local disk does not survive a session disconnect.
from google.colab import drive
drive.mount('/content/drive')

### Repo setup

This repo is public, so cloning it needs no GitHub personal access token (PAT) or login — a plain HTTPS clone works. The cloned code itself is small and lives on Colab's local (ephemeral) disk; that's fine, since it's just reproduced from git on every fresh session. Only the *data* and *checkpoints* need Drive.

In [ ]:
# Clone the repo (public -- no PAT/auth needed) and move into it.
!git clone https://github.com/mohamad-755/magnet-african-tokenization.git
%cd magnet-african-tokenization

### Installing dependencies

`requirements.txt` lists this project's dependencies: torch, numpy, sentencepiece, pyyaml, tqdm. Colab's base image already ships a CUDA-enabled torch build, so this step mainly ensures sentencepiece/pyyaml/tqdm are present without disturbing the preinstalled GPU-enabled torch.

In [ ]:
# Install this project's dependencies.
!pip install -q -r requirements.txt

In [ ]:
# Defensive PYTHONPATH setup: `python -m src.training.train` resolves the
# `src` package relative to the current working directory, which %cd above
# already set correctly -- this just guards against a cell being re-run out
# of order (a common Colab footgun) by explicitly adding the repo root.
import os
import sys

REPO_ROOT = os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.environ["PYTHONPATH"] = REPO_ROOT + os.pathsep + os.environ.get("PYTHONPATH", "")
print("PYTHONPATH set to:", REPO_ROOT)

# Confirm a GPU is actually attached before starting a real training run --
# if this prints False, go set Runtime > Change runtime type > T4 GPU first.
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## Training Setup

**Model hyperparameters** (see `src/training/train.py`'s `TrainConfig` for defaults): `d_model=256`, `n_heads=4`, 2 tokenization-stage layers, 6 middle-stage layers, 2 final-stage layers -- the hourglass transformer architecture from `src/model/hourglass_transformer.py`. `batch_size=16`, `learning_rate=5e-5`, `warmup_ratio=0.1`, matching the paper's Appendix D.1 starting point. `total_steps=5000` here is a reduced, time-bounded run (not the paper's full training budget) chosen to fit this project's time constraints while still producing a real, inspectable checkpoint.

**Why uniform β=0.5:** `--beta-by-language` sets each language's own boundary predictor's target compression rate (the fraction of byte positions that should become segment boundaries -- see `src/model/losses.py`'s `binomial_regularizer`, implementing the paper's Eq. 3). Setting every one of the 9 values to 0.5 is deliberate, not a placeholder: it's the configuration this project validated as training-stable. A follow-up experiment computed each language's own target from its byte-to-word ratio (paper Eq. 4) and trained with those 9 *different* values instead -- that run's boundary predictor collapsed to predicting almost no real boundaries at all, regardless of `reg_weight`. That failure is reproduced in the **Known limitation** appendix at the end of this notebook, not here.

**Routing:** this uses per-language boundary-predictor routing (`src/model/magnet.py`'s `LanguageRoutedBoundaryPredictor` -- one predictor per language, this project's own extension of the paper's script-level design), not the paper's original script-level routing.

In [ ]:
# Full training run: per-language routing, uniform beta=0.5 across all 9
# languages (index-aligned with dataset.LANGUAGES: sw,zu,yo,ig,ha,ny,am,rw,wo).
# This is the project's validated, training-stable baseline configuration.
#
# NOTE: re-running this cell will overwrite checkpoints already saved at
# --checkpoint-dir below. If you want to preserve a prior run's checkpoints,
# change --checkpoint-dir to a new path first.
!python -m src.training.train \
  --data-root /content/drive/MyDrive/DATA_ROOT \
  --languages sw,zu,yo,ig,ha,ny,am,rw,wo \
  --max-seq-len 512 \
  --d-model 256 --n-heads 4 --n-layers-tokenization 2 --n-layers-middle 6 --n-layers-final 2 \
  --batch-size 16 \
  --learning-rate 5e-5 \
  --warmup-ratio 0.1 \
  --total-steps 5000 \
  --beta-by-language 0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5 \
  --reg-weight 1.0 \
  --checkpoint-dir /content/drive/MyDrive/DATA_ROOT/checkpoints_per_language \
  --log-every 50 \
  --eval-every 1000

### What to expect during training

Each logged line looks like:

```
step 1000 lang=sw lr=4.44e-05 lm_loss=1.9511 reg_loss=3.5167 elapsed=192.4s
```

- `lm_loss` is the next-token prediction loss -- this should trend down over training as the language model improves.
- `reg_loss` is the binomial regularizer (Eq. 3) pulling the boundary predictor toward its target compression rate -- for a stable run like this one, it should stay in a bounded, non-exploding range rather than growing or collapsing to a degenerate extreme.
- `lang=` cycles through all 9 languages in round-robin order (`src/training/train.py`'s `RoundRobinLanguageLoader`) -- one language per logged step, so losses are attributable per-language, not a blended batch average.
- Every `--eval-every` steps, a block of `eval step N lang=... lm_loss=... reg_loss=...` lines runs a no-gradient pass over each language's held-out `eval_holdout` set.
- Every `--checkpoint-every` steps (and once more at the end), a `checkpoint saved: ...` line confirms a checkpoint was written to `--checkpoint-dir` on Drive.

With `total_steps=5000` on a T4, expect roughly 15-20 minutes end to end (based on this project's prior runs at this configuration).

## Inspection & Analysis

Once training finishes, `inspect_segmentation.py` loads the final checkpoint, prints qualitative examples per language with predicted segment boundaries marked `||` (paper Table 4 style), and reports the actual equitability metric -- average bytes/segment across each language's *full* eval set, not just the printed examples.

In [ ]:
# Load the trained checkpoint and inspect its segmentation behavior:
# qualitative ||-marked examples per language, plus the full eval_holdout
# equitability stats (avg segments/example, avg bytes/segment per language).
!python -m src.eval.inspect_segmentation \
  --checkpoint-path /content/drive/MyDrive/DATA_ROOT/checkpoints_per_language/step_4999.pt \
  --data-root /content/drive/MyDrive/DATA_ROOT \
  --languages sw,zu,yo,ig,ha,ny,am,rw,wo

## Interpreting the results

- **bytes/segment** is the average number of raw UTF-8 bytes covered by each predicted MAGNET segment for a language, computed as `total real bytes / total predicted segments` across the full eval set. Higher = coarser segmentation (fewer, longer segments); lower = finer/more fragmented segmentation (more, shorter segments). There's no single "correct" value -- the question this project cares about is whether it's *comparable across languages*, not any specific number.
- **Coefficient of variation (CV)** = population stdev / mean of bytes/segment across the 9 languages -- the fairness metric used throughout this project. Lower CV means more consistent segmentation granularity across languages (closer to equitable); higher CV means some languages are being segmented much more coarsely or finely than others.
- **Comparing script-level vs. per-language routing:** `results/per_language_vs_script_level_comparison.json` in the repo has the real numbers from both architectures at uniform β=0.5. The headline finding: per-language routing did *not* improve cross-language CV relative to script-level routing -- isolating just the 8 Latin-script languages (removing Amharic's confound, since it had a dedicated predictor either way), per-language routing's CV was nearly 3x *worse*. Sharing one predictor across languages appears to act as an implicit consistency mechanism that independent per-language predictors don't get for free.
- **Comparing against a BPE baseline:** `results/bpe_baseline_stats.json` has the same bytes/segment and CV metrics for a from-scratch-trained SentencePiece BPE tokenizer per language. MAGNET's raw byte-level boundaries can split individual multi-byte UTF-8 characters (severely for Amharic/Ge'ez -- see the `mid_char_splits` counts this notebook's inspection output prints); BPE cannot, by construction, since it tokenizes decoded Unicode text rather than raw bytes.
- See `README.md`'s **Project status** section for the full narrative across all three comparisons.

## Troubleshooting

**Colab disconnects mid-training.** Checkpoints are saved to Drive every `--checkpoint-every` steps plus once at the end, so a disconnect loses at most that many steps. Resume with `--resume /content/drive/MyDrive/DATA_ROOT/<checkpoint-dir>/latest.pt`, and **keep `--total-steps` and `--warmup-ratio` identical** to the original run -- only the step counter is restored from the checkpoint, not the learning-rate schedule's shape (PyTorch's `LambdaLR` can't pickle the lambda that defines it), so changing those two values on resume produces a different, inconsistent schedule.

**Drive read failures / hangs on large files.** This project hit a real, reproducible issue where Google Drive's virtual filesystem failed to read Hausa's 252MB `corpus.txt` directly -- even after Drive showed it as fully "synced" -- with errors like `OSError: [Errno 22] Invalid argument` or "Insufficient system resources exist to complete the requested service." If training or inspection hangs specifically while loading Hausa's data, this is almost certainly the same issue. There is no code-side fix that reliably worked around it (chunked reads didn't help); what worked was ensuring the specific file was fully re-synced/cached in Drive before reading. This repo's `data/raw/` folder is reserved for a local copy of that file as a fallback, but -- being 252MB -- it's gitignored and won't be present in a fresh Colab clone; you would need to copy it from Drive to Colab's local disk yourself if you hit this.

**Authentication.** The repo is public, so `git clone` needs no PAT -- if it fails, check your Colab instance's general internet connectivity, not GitHub credentials. If Drive mount fails or times out, re-run the mount cell and approve the browser permission prompt again; a stale mount from a previous session can also cause this.

**No GPU / `CUDA available: False`.** Go to `Runtime > Change runtime type` and select a GPU (T4 is sufficient), then re-run from the top. Training this configuration on CPU is impractical (expect an order of magnitude or more slower than the ~15-20 minutes on GPU).

**Wrong number of `--beta-by-language` values.** `train.py` validates this explicitly and will exit immediately with a clear error rather than crash deep inside training if you pass anything other than exactly 9 comma-separated values, index-aligned with `sw,zu,yo,ig,ha,ny,am,rw,wo`.

---

## Known limitation (documented failure -- not a recommended workflow)

The cells below reproduce a training run that **does not work**, kept here for transparency and reproducibility rather than as guidance. A follow-up experiment computed each language's own target compression rate from its byte-to-word ratio (paper Eq. 4, via `src/eval/compute_beta.py`) -- rather than the uniform β=0.5 used above -- and trained with those 9 distinct values. The boundary predictor collapsed to predicting almost no real boundaries at all (average segments per 512-byte example ≈ 1.00 for every language, i.e. essentially the entire chunk became one segment), uniformly across all 9 languages despite their quite different individual targets -- evidence of a shared optimization failure, not genuine per-language convergence. Reducing `reg_weight` 10x (1.0 → 0.1) did **not** resolve it. This remains unresolved; see `README.md`'s **Limitations / known open issues** section.

Run these only if you want to reproduce the failure itself (e.g. to debug it further) -- they are not part of the recommended pipeline above.

In [ ]:
# DOCUMENTED FAILURE -- do not treat this as a working configuration.
# Per-language beta targets from paper Eq. 4 (compute_beta.py), reg_weight=1.0.
# This collapses: every language's boundary predictor converges to predicting
# almost no real boundaries (see inspection output in the next cell).
!python -m src.training.train \
  --data-root /content/drive/MyDrive/DATA_ROOT \
  --languages sw,zu,yo,ig,ha,ny,am,rw,wo \
  --max-seq-len 512 \
  --d-model 256 --n-heads 4 --n-layers-tokenization 2 --n-layers-middle 6 --n-layers-final 2 \
  --batch-size 16 \
  --learning-rate 5e-5 \
  --warmup-ratio 0.1 \
  --total-steps 5000 \
  --beta-by-language 0.1571,0.1069,0.1469,0.1572,0.1730,0.1423,0.0810,0.1381,0.2081 \
  --reg-weight 1.0 \
  --checkpoint-dir /content/drive/MyDrive/DATA_ROOT/checkpoints_per_language_tuned_beta \
  --log-every 50 \
  --eval-every 1000

In [ ]:
# Inspecting the collapsed checkpoint: expect avg_segments/example close to
# 1.00 for every language, confirming the collapse described above.
!python -m src.eval.inspect_segmentation \
  --checkpoint-path /content/drive/MyDrive/DATA_ROOT/checkpoints_per_language_tuned_beta/step_4999.pt \
  --data-root /content/drive/MyDrive/DATA_ROOT \
  --languages sw,zu,yo,ig,ha,ny,am,rw,wo

In [ ]:
# DOCUMENTED FAILURE, attempt 2 -- reg_weight lowered 10x (1.0 -> 0.1) to test
# whether the regularizer was simply too strong. It was not sufficient: this
# also collapses, to bit-identical eval stats as the reg_weight=1.0 attempt
# above (expected once real segmentation is fully gone -- avg bytes/segment
# becomes a pure function of the data, not the model, at that point).
!python -m src.training.train \
  --data-root /content/drive/MyDrive/DATA_ROOT \
  --languages sw,zu,yo,ig,ha,ny,am,rw,wo \
  --max-seq-len 512 \
  --d-model 256 --n-heads 4 --n-layers-tokenization 2 --n-layers-middle 6 --n-layers-final 2 \
  --batch-size 16 \
  --learning-rate 5e-5 \
  --warmup-ratio 0.1 \
  --total-steps 5000 \
  --beta-by-language 0.1571,0.1069,0.1469,0.1572,0.1730,0.1423,0.0810,0.1381,0.2081 \
  --reg-weight 0.1 \
  --checkpoint-dir /content/drive/MyDrive/DATA_ROOT/checkpoints_per_language_tuned_beta_lowreg \
  --log-every 50 \
  --eval-every 1000

In [ ]:
# Inspecting the reg_weight=0.1 attempt's checkpoint -- confirms the same collapse.
!python -m src.eval.inspect_segmentation \
  --checkpoint-path /content/drive/MyDrive/DATA_ROOT/checkpoints_per_language_tuned_beta_lowreg/step_4999.pt \
  --data-root /content/drive/MyDrive/DATA_ROOT \
  --languages sw,zu,yo,ig,ha,ny,am,rw,wo

### What we take from this

Cutting `reg_weight` 10x did not change the outcome at all -- both attempts produced bit-identical eval statistics, which is itself informative: once a boundary predictor fully collapses to zero real boundaries, `avg_bytes/segment` becomes a pure function of the data (total bytes divided by number of examples), completely independent of model weights. That both runs landed there regardless of `reg_weight` suggests the fix isn't a matter of scalar reweighting alone -- likely candidates for future work include a warmup schedule for the regularizer itself (rather than full weight from step 0), or per-language `reg_weight` scaling proportional to each target's distance from the ~0.5 initialization point. Neither was tested here due to time constraints.